# Test-Set Posterior Workbench

This notebook loads a trained SimFormer checkpoint and held-out test rows, then lets you:

1. Filter candidate stars by physical `logAge` and `m_init` ranges.
2. Sample posteriors for selected stars.
3. Inspect per-star posterior summaries, including:
   - posterior mean `rad`
   - posterior mean `logAge`
   - `P(logAge < 7.8)` (younger than ~60 Myr)
4. Plot corner-style posterior panels for chosen stars.
5. Run a TARP calibration check on the sampled subset.


In [ ]:
import sys
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Make imports robust when this notebook is opened from notebooks/.
def _resolve_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/Users/ratzenboe/Documents/work/projects/sbi_variants'),
    ]
    for c in candidates:
        if (c / 'sample_mock_galaxy.py').exists() and (c / 'eval_utils.py').exists():
            return c.resolve()
    raise FileNotFoundError('Could not locate repo root containing sample_mock_galaxy.py and eval_utils.py')

REPO_ROOT = _resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sample_mock_galaxy import sample_posterior as sample_posterior_legacy
from eval_utils import (
    load_cache_arrays,
    to_input_tensors,
    column_indices,
    projection_ranks,
    central_rank_coverage,
    ks_uniform,
)
from inference_utils import NormStats
from sample_sbi_posterior import (
    _build_model_from_config,
    _load_json as _load_sbi_json,
    _load_state_dict as _load_sbi_state_dict,
    _prepare_from_cache as _prepare_cache_for_posterior,
    _resolve_color_definitions,
    _resolve_input_layout,
)


def load_model_compat(model_dir: str, run_name: str = "default", device: str = "cpu"):
    """Load either legacy Simformer artifacts or direct-SBI posterior artifacts."""
    from sample_mock_galaxy import load_model as load_model_legacy

    legacy_cfg = os.path.join(model_dir, f"model_config_{run_name}.json")
    posterior_cfg = os.path.join(model_dir, f"posterior_config_{run_name}.json")

    # Legacy Simformer artifacts
    if os.path.exists(legacy_cfg):
        model, norm_stats = load_model_legacy(model_dir, run_name=run_name, device=device)
        meta = {
            "kind": "legacy",
            "method": "simformer",
            "theta_columns": [],
            "input_columns_base": list(norm_stats.columns),
            "input_columns_model": list(norm_stats.columns),
            "use_colors": False,
            "color_definitions": [],
            "color_means": None,
            "color_stds": None,
        }
        return model, norm_stats, meta

    # New direct-SBI posterior artifacts
    if not os.path.exists(posterior_cfg):
        raise FileNotFoundError(
            f"No config found for run '{run_name}' in {model_dir}. "
            f"Expected either {legacy_cfg} or {posterior_cfg}."
        )

    config = _load_sbi_json(posterior_cfg)
    ckpt_path = os.path.join(model_dir, f"best_model_{run_name}.pt")
    meta_path = os.path.join(model_dir, f"posterior_norm_meta_{run_name}.npz")
    if not os.path.exists(meta_path):
        fallback_meta = os.path.join(model_dir, "norm_stats.npz")
        if os.path.exists(fallback_meta):
            meta_path = fallback_meta
        else:
            raise FileNotFoundError(
                f"No normalization metadata found at {meta_path} or {fallback_meta}."
            )

    norm_stats = NormStats(meta_path)
    (
        input_columns_base,
        input_columns_model,
        use_colors,
        color_names,
        color_means,
        color_stds,
    ) = _resolve_input_layout(config, norm_stats)

    model = _build_model_from_config(config, input_columns_override=input_columns_model)
    state_dict = _load_sbi_state_dict(ckpt_path, device=device)
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    color_definitions = _resolve_color_definitions(color_names) if use_colors else []
    meta = {
        "kind": "posterior",
        "method": str(config.get("method", "flow_matching")),
        "theta_columns": [str(c) for c in config["theta_columns"]],
        "input_columns_base": list(input_columns_base),
        "input_columns_model": list(input_columns_model),
        "use_colors": bool(use_colors),
        "color_definitions": color_definitions,
        "color_means": color_means,
        "color_stds": color_stds,
    }
    return model, norm_stats, meta


@torch.no_grad()
def sample_posterior_compat(
    *,
    model,
    model_meta,
    condition_values,
    condition_mask,
    observed_mask,
    errors,
    num_samples,
    batch_size,
    steps,
    device,
):
    """Sample posterior for either legacy Simformer or new direct-SBI models."""
    kind = model_meta.get("kind", "legacy")
    if kind == "posterior":
        values = condition_values.to(device)
        errs = errors.to(device)
        obs = observed_mask.to(device)
        method = model_meta.get("method", "flow_matching")
        if method == "flow_matching":
            out = model.sample(
                values=values,
                errors=errs,
                observed_mask=obs,
                num_samples=num_samples,
                steps=steps,
            )
        elif method in ("normalizing_flow", "realnvp"):
            out = model.sample(
                values=values,
                errors=errs,
                observed_mask=obs,
                num_samples=num_samples,
            )
        else:
            raise ValueError(f"Unsupported posterior method: {method}")
        return out.cpu()

    return sample_posterior_legacy(
        model=model,
        condition_values=condition_values,
        condition_mask=condition_mask,
        observed_mask=observed_mask,
        errors=errors,
        num_samples=num_samples,
        batch_size=batch_size,
        steps=steps,
        device=device,
    )


plt.rcParams["figure.dpi"] = 120
print(f"REPO_ROOT: {REPO_ROOT}")



In [ ]:
# -----------------------------
# Required paths
# -----------------------------
MODEL_DIR = Path("/absolute/path/to/model_output")
RUN_NAME = "default"
CACHE_PATH = Path("/absolute/path/to/build_arrays_cache.npz")
INDEX_FILE = Path("/absolute/path/to/test_indices.npy")  # set to None to use all cache rows

# -----------------------------
# Runtime + reproducibility
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# -----------------------------
# Star selection in physical units
# -----------------------------
LOGAGE_RANGE = (None, None)      # e.g. (6.5, 8.2)
MASS_RANGE = (None, None)        # e.g. (0.8, 2.0)
MAX_FILTERED_STARS = None        # optional cap after filtering

# Which stars to sample from filtered pool
N_STARS_TO_SAMPLE = 16
STAR_PICK_MODE = "random"       # "random" | "head" | "manual"
MANUAL_STAR_GLOBAL_INDICES = []  # used only if STAR_PICK_MODE == "manual"

# -----------------------------
# Posterior sampling
# -----------------------------
NUM_POSTERIOR_SAMPLES = 512
STEPS = 128
BATCH_SIZE = 256

# -----------------------------
# Science outputs
# -----------------------------
YOUNG_LOGAGE_THRESHOLD = 7.8  # ~60 Myr
TARGET_COLS = ["feh", "m_init", "logAge", "rad", "logL", "logT", "logg", "Av"]
CORNER_COLS = ["logAge", "m_init", "rad", "Av"]
N_CORNER_STARS = 3

# -----------------------------
# TARP
# -----------------------------
RUN_TARP = True
TARP_NUM_PROJECTIONS = 256
TARP_ALPHA_GRID = np.linspace(0.1, 0.9, 9)

# -----------------------------
# Outputs
# -----------------------------
OUTPUT_DIR = MODEL_DIR / "eval_notebook_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DEVICE: {DEVICE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

assert MODEL_DIR.exists(), f"MODEL_DIR not found: {MODEL_DIR}"
assert CACHE_PATH.exists(), f"CACHE_PATH not found: {CACHE_PATH}"
if INDEX_FILE is not None:
    assert INDEX_FILE.exists(), f"INDEX_FILE not found: {INDEX_FILE}"

print("Loading model...")
model, norm_stats, model_meta = load_model_compat(str(MODEL_DIR), run_name=RUN_NAME, device=DEVICE)

print("Loading cache rows...")
if model_meta["kind"] == "posterior":
    values_model, errors_model, observed_model, selected_idx = _prepare_cache_for_posterior(
        cache_path=str(CACHE_PATH),
        input_columns=model_meta["input_columns_base"],
        index_file=str(INDEX_FILE) if INDEX_FILE is not None else None,
        max_stars=None,
        sample_mode="random",
        seed=SEED,
        use_colors=model_meta["use_colors"],
        color_definitions=model_meta["color_definitions"],
        color_means=model_meta["color_means"],
        color_stds=model_meta["color_stds"],
    )
    # Also load full cache columns for truth/filtering diagnostics.
    values_norm, errors_norm, observed_mask, selected_idx_full = load_cache_arrays(
        cache_path=str(CACHE_PATH),
        index_file=str(INDEX_FILE) if INDEX_FILE is not None else None,
        max_stars=None,
        sample_mode="random",
        seed=SEED,
        expected_columns=norm_stats.columns,
    )
    if not np.array_equal(selected_idx, selected_idx_full):
        raise RuntimeError("Selected row mismatch between posterior input loader and eval cache loader.")

    cv, cm, om, er = to_input_tensors(
        values_model,
        errors_model,
        observed_model,
        columns=model_meta["input_columns_model"],
        device="cpu",
    )
else:
    values_norm, errors_norm, observed_mask, selected_idx = load_cache_arrays(
        cache_path=str(CACHE_PATH),
        index_file=str(INDEX_FILE) if INDEX_FILE is not None else None,
        max_stars=None,
        sample_mode="random",
        seed=SEED,
        expected_columns=norm_stats.columns,
    )

    cv, cm, om, er = to_input_tensors(
        values_norm,
        errors_norm,
        observed_mask,
        columns=norm_stats.columns,
        device="cpu",
    )

print(f"Loaded rows: {len(selected_idx):,}")
print(f"Artifact kind: {model_meta['kind']} | method: {model_meta['method']}")
if model_meta["kind"] == "posterior":
    n_base = len(model_meta["input_columns_base"])
    n_model = len(model_meta["input_columns_model"])
    print(f"Posterior input columns: base={n_base}, total_model_inputs={n_model}")
    print(f"Posterior theta columns: {model_meta['theta_columns']}")
print(f"Num full columns in norm_stats: {len(norm_stats.columns)}")



In [ ]:
def apply_range(df: pd.DataFrame, col: str, bounds):
    lo, hi = bounds
    out = df
    if col not in out.columns:
        return out
    if lo is not None:
        out = out[out[col] >= lo]
    if hi is not None:
        out = out[out[col] <= hi]
    return out

needed_cols = sorted(set(TARGET_COLS + ["logAge", "m_init", "rad", "Av"]))
available_cols = [c for c in needed_cols if c in norm_stats.columns]

if not available_cols:
    raise ValueError("None of the requested filter/target columns exist in norm_stats.columns")

available_idx = [norm_stats.columns.index(c) for c in available_cols]
truth_phys_available = norm_stats.denormalize_numpy(values_norm[:, available_idx], column_indices=available_idx)

pool_df = pd.DataFrame(truth_phys_available, columns=available_cols)
pool_df["row_in_loaded"] = np.arange(len(selected_idx), dtype=np.int64)
pool_df["global_index"] = selected_idx

filtered_df = apply_range(pool_df, "logAge", LOGAGE_RANGE)
filtered_df = apply_range(filtered_df, "m_init", MASS_RANGE)
if MAX_FILTERED_STARS is not None and len(filtered_df) > MAX_FILTERED_STARS:
    filtered_df = filtered_df.head(MAX_FILTERED_STARS).copy()

print(f"Pool rows:     {len(pool_df):,}")
print(f"Filtered rows: {len(filtered_df):,}")
print(f"Applied LOGAGE_RANGE={LOGAGE_RANGE}, MASS_RANGE={MASS_RANGE}")

display_cols = [c for c in ["global_index", "row_in_loaded", "logAge", "m_init", "rad", "Av"] if c in filtered_df.columns]
display(filtered_df[display_cols].head(10))


In [ ]:
if filtered_df.empty:
    raise ValueError("No stars left after filtering. Relax LOGAGE_RANGE/MASS_RANGE.")

if STAR_PICK_MODE == "manual":
    if not MANUAL_STAR_GLOBAL_INDICES:
        raise ValueError("STAR_PICK_MODE='manual' but MANUAL_STAR_GLOBAL_INDICES is empty.")
    chosen_df = filtered_df[filtered_df["global_index"].isin(MANUAL_STAR_GLOBAL_INDICES)].copy()
elif STAR_PICK_MODE == "head":
    chosen_df = filtered_df.head(N_STARS_TO_SAMPLE).copy()
elif STAR_PICK_MODE == "random":
    n_take = min(N_STARS_TO_SAMPLE, len(filtered_df))
    chosen_df = filtered_df.sample(n=n_take, random_state=SEED).copy()
else:
    raise ValueError(f"Unsupported STAR_PICK_MODE: {STAR_PICK_MODE}")

if chosen_df.empty:
    raise ValueError("No stars selected for sampling.")

chosen_df = chosen_df.sort_values("global_index").reset_index(drop=True)
chosen_rows = chosen_df["row_in_loaded"].to_numpy(dtype=np.int64)
chosen_global_idx = chosen_df["global_index"].to_numpy(dtype=np.int64)

print(f"Chosen stars: {len(chosen_rows)}")
display_cols = [c for c in ["global_index", "logAge", "m_init", "rad", "Av"] if c in chosen_df.columns]
display(chosen_df[display_cols])


In [ ]:
target_cols_available = [c for c in TARGET_COLS if c in norm_stats.columns]
if not target_cols_available:
    raise ValueError("None of TARGET_COLS are present in model columns.")

if model_meta["kind"] == "posterior":
    theta_cols = list(model_meta["theta_columns"])
    target_cols_available = [c for c in target_cols_available if c in theta_cols]
    if not target_cols_available:
        raise ValueError(
            "None of TARGET_COLS are in posterior theta_columns. "
            f"theta_columns={theta_cols}"
        )
    target_idx_model = [theta_cols.index(c) for c in target_cols_available]
    target_idx_norm = column_indices(target_cols_available, columns=norm_stats.columns)
else:
    target_idx = column_indices(target_cols_available, columns=norm_stats.columns)

print("Sampling posteriors...")
samples_norm = sample_posterior_compat(
    model=model,
    model_meta=model_meta,
    condition_values=cv[chosen_rows],
    condition_mask=cm[chosen_rows],
    observed_mask=om[chosen_rows],
    errors=er[chosen_rows],
    num_samples=NUM_POSTERIOR_SAMPLES,
    batch_size=BATCH_SIZE,
    steps=STEPS,
    device=DEVICE,
).numpy()

if model_meta["kind"] == "posterior":
    truth_norm = values_norm[chosen_rows][:, target_idx_norm]
    samples_phys = norm_stats.denormalize_numpy(samples_norm[:, :, target_idx_model], column_indices=target_idx_norm)
    truth_phys = norm_stats.denormalize_numpy(truth_norm, column_indices=target_idx_norm)
else:
    truth_norm = values_norm[chosen_rows][:, target_idx]
    samples_phys = norm_stats.denormalize_numpy(samples_norm[:, :, target_idx], column_indices=target_idx)
    truth_phys = norm_stats.denormalize_numpy(truth_norm, column_indices=target_idx)

print(f"samples_phys shape: {samples_phys.shape} (N_stars, N_samples, N_targets)")
print(f"target columns: {target_cols_available}")



In [ ]:
col_to_j = {c: j for j, c in enumerate(target_cols_available)}
rows = []
for i in range(samples_phys.shape[0]):
    row = {
        "global_index": int(chosen_global_idx[i]),
        "row_in_loaded": int(chosen_rows[i]),
    }

    if "rad" in col_to_j:
        j = col_to_j["rad"]
        row["posterior_mean_rad"] = float(samples_phys[i, :, j].mean())
        row["truth_rad"] = float(truth_phys[i, j])

    if "logAge" in col_to_j:
        j = col_to_j["logAge"]
        la = samples_phys[i, :, j]
        row["posterior_mean_logAge"] = float(la.mean())
        row["truth_logAge"] = float(truth_phys[i, j])
        row[f"p_logAge_lt_{YOUNG_LOGAGE_THRESHOLD}"] = float((la < YOUNG_LOGAGE_THRESHOLD).mean())

    if "m_init" in col_to_j:
        j = col_to_j["m_init"]
        row["truth_m_init"] = float(truth_phys[i, j])

    rows.append(row)

summary_df = pd.DataFrame(rows)
prob_col = f"p_logAge_lt_{YOUNG_LOGAGE_THRESHOLD}"
if prob_col in summary_df.columns:
    summary_df = summary_df.sort_values(prob_col, ascending=False).reset_index(drop=True)

display(summary_df)

summary_csv = OUTPUT_DIR / "posterior_summary_selected_stars.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Saved summary table: {summary_csv}")


In [ ]:
def plot_corner_like(samples_star: np.ndarray, cols: list[str], truth_star: np.ndarray | None = None, title: str | None = None):
    d = len(cols)
    fig, axes = plt.subplots(d, d, figsize=(2.2 * d, 2.2 * d), squeeze=False)
    for i in range(d):
        for j in range(d):
            ax = axes[i, j]
            if i < j:
                ax.axis("off")
                continue
            if i == j:
                ax.hist(samples_star[:, j], bins=40, density=True, alpha=0.8)
                if truth_star is not None:
                    ax.axvline(truth_star[j], color="tab:red", lw=1.5)
            else:
                ax.scatter(samples_star[:, j], samples_star[:, i], s=3, alpha=0.08)
                if truth_star is not None:
                    ax.scatter([truth_star[j]], [truth_star[i]], s=30, c="tab:red", marker="x")
            if i == d - 1:
                ax.set_xlabel(cols[j])
            else:
                ax.set_xticklabels([])
            if j == 0 and i > 0:
                ax.set_ylabel(cols[i])
            elif j > 0:
                ax.set_yticklabels([])
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()

corner_cols_available = [c for c in CORNER_COLS if c in target_cols_available]
if len(corner_cols_available) < 2:
    print("Need at least 2 available CORNER_COLS for corner-style plotting.")
else:
    corner_j = [target_cols_available.index(c) for c in corner_cols_available]
    n_plot = min(N_CORNER_STARS, samples_phys.shape[0])
    for i in range(n_plot):
        title = f"global_index={int(chosen_global_idx[i])}"
        plot_corner_like(
            samples_phys[i][:, corner_j],
            corner_cols_available,
            truth_star=truth_phys[i][corner_j],
            title=title,
        )


In [ ]:
if RUN_TARP:
    # Use whichever requested target columns are available in this checkpoint.
    tarp_cols = list(target_cols_available)
    tarp_j = [target_cols_available.index(c) for c in tarp_cols]

    samples_t = samples_phys[:, :, tarp_j]
    truth_t = truth_phys[:, tarp_j]

    u = projection_ranks(
        samples=samples_t,
        truth=truth_t,
        num_projections=TARP_NUM_PROJECTIONS,
        seed=SEED,
    )
    u_flat = u.reshape(-1)

    rows = []
    for alpha in TARP_ALPHA_GRID:
        emp = central_rank_coverage(u_flat, float(alpha))
        rows.append(
            {
                "alpha": float(alpha),
                "empirical_coverage": float(emp),
                "calibration_error": float(emp - alpha),
            }
        )

    tarp_df = pd.DataFrame(rows).sort_values("alpha").reset_index(drop=True)
    ks = ks_uniform(u_flat)
    ace = float(np.mean(np.abs(tarp_df["calibration_error"].values)))

    display(tarp_df)
    print(f"KS(uniform): {ks:.4f}")
    print(f"TARP ACE:    {ace:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax = axes[0]
    ax.plot(tarp_df["alpha"], tarp_df["empirical_coverage"], marker="o", lw=2, label="empirical")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="ideal")
    ax.set_xlabel("Nominal central coverage")
    ax.set_ylabel("Empirical central coverage")
    ax.set_title("TARP Calibration Curve")
    ax.grid(alpha=0.3)
    ax.legend()

    ax = axes[1]
    ax.hist(u_flat, bins=30, density=True, alpha=0.85)
    ax.axhline(1.0, color="k", linestyle="--", lw=1)
    ax.set_xlabel("Rank u")
    ax.set_ylabel("Density")
    ax.set_title("TARP Rank Histogram")
    ax.grid(alpha=0.3)

    fig.tight_layout()
    plt.show()

    tarp_curve_csv = OUTPUT_DIR / "tarp_curve_selected_stars.csv"
    tarp_summary_json = OUTPUT_DIR / "tarp_summary_selected_stars.json"
    tarp_df.to_csv(tarp_curve_csv, index=False)
    with open(tarp_summary_json, "w") as f:
        json.dump(
            {
                "num_stars": int(samples_phys.shape[0]),
                "num_samples_per_star": int(samples_phys.shape[1]),
                "num_projections": int(TARP_NUM_PROJECTIONS),
                "target_cols": tarp_cols,
                "ks_uniform": float(ks),
                "ace": float(ace),
            },
            f,
            indent=2,
        )
    print(f"Saved: {tarp_curve_csv}")
    print(f"Saved: {tarp_summary_json}")
else:
    print("RUN_TARP=False -> skipping TARP block")


## Notes

- This notebook evaluates calibration on the stars you actually sampled (`chosen_df`).
- To evaluate a larger subset, increase `N_STARS_TO_SAMPLE` (and keep an eye on runtime).
- `p_logAge_lt_7.8` is estimated via Monte Carlo posterior draws for each star.
